# OSFT Training Template

This notebook helps you get started with **Orthogonal Subspace Fine-Tuning (OSFT)** using Training Hub.
OSFT trains new knowledge while preserving existing model capabilities by constraining
updates to an orthogonal subspace of the weight matrices.

## What you need
1. A HuggingFace model path (or local model)
2. A training dataset in JSONL chat format
3. One or more GPUs

## What this notebook does
- Detects available GPUs and VRAM
- Calculates memory-safe values for `max_tokens_per_gpu`, accounting for OSFT overhead from `unfreeze_rank_ratio`
- Provides sensible defaults for learning rate, batch size, and epochs
- Launches training with a single call to `training_hub.osft()`

## Setup

In [ ]:
# Install training-hub with CUDA support (two-step install required)
# Skip if already installed
# %pip install training-hub
# %pip install training-hub[cuda] --no-build-isolation

## 1. User Inputs

Configure these values. Everything else is calculated automatically.

In [ ]:
# === REQUIRED: Set these ===
MODEL_PATH = "ibm-granite/granite-3.3-8b-instruct"  # HuggingFace model or local path
DATA_PATH = "./train_data.jsonl"                     # Path to your JSONL training data
CKPT_OUTPUT_DIR = "./osft_checkpoints"               # Where to save checkpoints

# === OPTIONAL: Hardware overrides ===
NUM_GPUS = None       # Auto-detected if None
NUM_NODES = 1         # Number of nodes (default: 1)
VRAM_PER_GPU = None   # Auto-detected if None (in GB, e.g. 80)

# === OSFT-specific ===
UNFREEZE_RANK_RATIO = 0.5  # Recommended default; lower = more preservation, higher = more learning

## 2. Hardware Detection

In [ ]:
import torch
from transformers import AutoConfig

# Detect GPUs
if NUM_GPUS is None:
    NUM_GPUS = torch.cuda.device_count()
    assert NUM_GPUS > 0, "No GPUs detected. Set NUM_GPUS manually."

# Detect VRAM
if VRAM_PER_GPU is None:
    props = torch.cuda.get_device_properties(0)
    VRAM_PER_GPU = props.total_memory / (1024**3)

gpu_name = torch.cuda.get_device_name(0)
print(f"GPU: {gpu_name}")
print(f"GPUs: {NUM_GPUS}")
print(f"Nodes: {NUM_NODES}")
print(f"VRAM per GPU: {VRAM_PER_GPU:.1f} GB")
print(f"Total VRAM: {NUM_GPUS * NUM_NODES * VRAM_PER_GPU:.1f} GB")

## 3. Calculate Safe Hyperparameters

OSFT has the same base memory profile as SFT (~16 bytes/param) but adds overhead
from the SVD computation and unfrozen subspace. Higher `unfreeze_rank_ratio` means
more memory. The scaling is roughly:
- ratio=0 → ~0.5x SFT optimizer memory
- ratio=0.33 → ~1x SFT (equivalent)
- ratio=1.0 → ~2x SFT optimizer memory

OSFT also uses Liger Kernels by default, which reduces activation memory.

In [ ]:
# Load model config (lightweight — no model download)
config = AutoConfig.from_pretrained(MODEL_PATH, trust_remote_code=True)
num_params = getattr(config, "num_parameters", None)

if num_params is None:
    hidden = config.hidden_size
    vocab = config.vocab_size
    layers = config.num_hidden_layers
    intermediate = getattr(config, "intermediate_size", hidden * 4)
    n_heads = config.num_attention_heads
    n_kv = getattr(config, "num_key_value_heads", n_heads)
    head_dim = hidden // n_heads
    # Attention (GQA-aware) + MLP (gate/up/down) per layer + embeddings
    attn = hidden * (n_heads + 2 * n_kv) * head_dim + hidden * hidden
    mlp = 3 * hidden * intermediate
    num_params = vocab * hidden + layers * (attn + mlp)
# OSFT memory scaling: 0.5 + 1.5 * unfreeze_rank_ratio
osft_ratio = 0.5 + (1.5 * UNFREEZE_RANK_RATIO)

# Base: param(4B) + grad(4B) + optimizer(8B scaled by osft_ratio)
bytes_per_param = 4 + 4 + (8 * osft_ratio)
base_memory_per_gpu_gb = (num_params * bytes_per_param) / (NUM_GPUS * 1024**3)

# Available VRAM for activations (Liger reduces activation memory, so less headroom needed)
usable_vram = VRAM_PER_GPU * 0.85
activation_budget_gb = max(usable_vram - base_memory_per_gpu_gb, 1.0)

# Activation cost per token (Liger kernels reduce this by ~30-40% vs standard)
hidden = config.hidden_size
layers = config.num_hidden_layers
liger_factor = 0.65  # Liger reduces activation memory
bytes_per_token = hidden * layers * 2 * liger_factor / NUM_GPUS
max_tokens_from_budget = int((activation_budget_gb * 1024**3) / bytes_per_token)

SAFE_MAX_TOKENS_PER_GPU = min(max(256, (max_tokens_from_budget // 256) * 256), 16384)
SAFE_MAX_SEQ_LEN = min(SAFE_MAX_TOKENS_PER_GPU, 4096)

print(f"\nModel: {MODEL_PATH}")
print(f"Estimated parameters: {num_params / 1e9:.2f}B")
print(f"OSFT memory scaling factor: {osft_ratio:.2f}x")
print(f"Base memory per GPU: {base_memory_per_gpu_gb:.1f} GB")
print(f"Activation budget per GPU: {activation_budget_gb:.1f} GB")
print(f"\nCalculated safe values:")
print(f"  max_tokens_per_gpu = {SAFE_MAX_TOKENS_PER_GPU}")
print(f"  max_seq_len = {SAFE_MAX_SEQ_LEN}")
print(f"  unfreeze_rank_ratio = {UNFREEZE_RANK_RATIO}")
print(f"  nproc_per_node = {NUM_GPUS}")

## 4. Training Configuration

These defaults work well for most OSFT tasks. Adjust as needed.

In [ ]:
# Training hyperparameters
LEARNING_RATE = 5e-6          # Same range as SFT: 5e-6 for small datasets, 1e-6 for large
EFFECTIVE_BATCH_SIZE = 64     # 32-64 for <1k samples, 128 for 1k-10k, 256+ for >10k
NUM_EPOCHS = 2                # 2-3 typical

print("Training config:")
print(f"  learning_rate = {LEARNING_RATE}")
print(f"  effective_batch_size = {EFFECTIVE_BATCH_SIZE}")
print(f"  num_epochs = {NUM_EPOCHS}")
print(f"  max_seq_len = {SAFE_MAX_SEQ_LEN}")
print(f"  max_tokens_per_gpu = {SAFE_MAX_TOKENS_PER_GPU}")
print(f"  unfreeze_rank_ratio = {UNFREEZE_RANK_RATIO}")
print(f"  nproc_per_node = {NUM_GPUS}")
if NUM_NODES > 1:
    print(f"  nnodes = {NUM_NODES}")

## 5. Launch Training

In [ ]:
from training_hub import osft

result = osft(
    model_path=MODEL_PATH,
    data_path=DATA_PATH,
    ckpt_output_dir=CKPT_OUTPUT_DIR,
    # Calculated values
    max_tokens_per_gpu=SAFE_MAX_TOKENS_PER_GPU,
    max_seq_len=SAFE_MAX_SEQ_LEN,
    nproc_per_node=NUM_GPUS,
    nnodes=NUM_NODES if NUM_NODES > 1 else None,
    # OSFT-specific
    unfreeze_rank_ratio=UNFREEZE_RANK_RATIO,
    # Training hyperparameters
    learning_rate=LEARNING_RATE,
    effective_batch_size=EFFECTIVE_BATCH_SIZE,
    num_epochs=NUM_EPOCHS,
)

print("Training complete!")

## 6. Visualize Training Loss

In [ ]:
from training_hub import plot_loss

plot_loss(CKPT_OUTPUT_DIR, ema=True)

## Tips

- **OOM?** Reduce `UNFREEZE_RANK_RATIO` (reduces SVD overhead) or lower `SAFE_MAX_TOKENS_PER_GPU`
- **Not learning enough?** Increase `UNFREEZE_RANK_RATIO` (up to 0.7 for small models)
- **Too much forgetting?** Decrease `UNFREEZE_RANK_RATIO` (0.1-0.3 for maximum preservation)
- **Knowledge data?** Add `unmask_messages=True` to unmask user messages for better knowledge ingestion
- **Target specific layers?** Use `target_patterns` to restrict OSFT to MLPs or attention only
- **Pretraining on documents?** Add `is_pretraining=True` and `block_size=2048` to the `osft()` call